In [20]:
from __future__ import annotations

from pathlib import Path
from typing import Any, TypedDict, List, Optional

from langgraph.graph import StateGraph, START, END
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

from pydantic import BaseModel, Field
import os


from core import *

In [21]:
from core import LF_CONFIG, LF_CONFIG_LANGGRAPH_FANOUT, LLM_BASIC, RETRY

LF_CONFIG

{'callbacks': [<langfuse.langchain.CallbackHandler.LangchainCallbackHandler at 0x72a31a3f1850>]}

In [22]:
import langchain
langchain.__version__

'1.2.13'

In [23]:
class SMILES_Data(BaseModel):  # Phase of the molecule
    SMILES_columns: List[str] = Field(default_factory=list, description="Columns in csv file that contain SMILES")

class SMILESandCSV_Data(SMILES_Data):
    CSV_file_name: str = Field(description="Name of the CSV file.")


class EssentialColumnAnalysis(BaseModel):
    file_and_columns: List[SMILESandCSV_Data]


In [24]:
detect_smiles_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a chemistry data assistant. Detect the CSV column that contains SMILES and identify rows that contain valid-looking SMILES.",
    ),
    (
        "user",
        """
Return JSON with exactly these keys:
- column_SMILES: string | null
- smiles_rows: list[int]

Rules:
- Use only column names from the provided CSV preview.
- Detect SMILES-like values (typical symbols include letters, digits, =, #, (, ), [, ], +, -, @, /, \\\\).
- If no column is clearly SMILES, set column_SMILES to null and smiles_rows to [].
- smiles_rows must be 0-based row indices from the provided preview rows only.
- Do not invent rows or columns.

CSV file name:
{input_csv}

CSV preview:
{csv_preview}
""".strip(),
    ),
])

In [25]:
# To Agent: do not modify.
def store_state(state: DatasetTagsDataState):
    key_folder = Path(state.workspace_path) / state.key
    VersionedYaml(key_folder).create("tags", state.model_dump())

In [26]:
KEY = "ASPECT" #"ALGALTOX"
d,_ = VersionedYaml(Path(WORKSPACE) / KEY).read("detect_files")
d

{'workspace_path': '/home/kostanew/2026/CDGReview/workspace',
 'key': 'ASPECT',
 'list_files': ['README.md', 'chemicals.csv'],
 'description_files': ['README.md'],
 'data_files': ['chemicals.csv'],
 'other_files': []}

In [27]:
all_keys = sorted(
    p.name
    for p in Path(WORKSPACE).iterdir()
    if p.is_dir() and not p.name.startswith(".")
)



In [28]:
from pathlib import Path
import pandas as pd

ROOT = Path(WORKSPACE)
EXCEL_SUFFIXES = {".xls", ".xlsx"}


def export_excel_sheets_to_csv(root: Path):
    excel_files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in EXCEL_SUFFIXES]

    exported = 0
    broken_files = []

    for excel_path in excel_files:
        try:
            sheets = pd.read_excel(excel_path, sheet_name=None)
        except Exception as exc:
            broken_files.append((excel_path, str(exc)))
            continue

        k = 1
        for _, df in sheets.items():
            out_path = excel_path.parent / f"data{k}.csv"
            while out_path.exists():
                k += 1
                out_path = excel_path.parent / f"data{k}.csv"

            df.to_csv(out_path, index=False)
            exported += 1
            k += 1

    print(f"Excel files found: {len(excel_files)}")
    print(f"CSV files exported: {exported}")
    print(f"Broken Excel files: {len(broken_files)}")

    for path, reason in broken_files:
        print(f"Broken file: {path}")
        print(f"  Reason: {reason}")

export_excel_sheets_to_csv(ROOT)

Excel files found: 9
CSV files exported: 46
Broken Excel files: 0


In [29]:
import pandas as pd

missing_paths: list[Path] = []
non_csv_paths: list[Path] = []
broken_csv_paths: list[Path] = []

for key in all_keys:
    key_dir = Path(WORKSPACE) / key
    detect_files_state, _ = VersionedYaml(key_dir).read("detect_files")

    for file_name in detect_files_state.get("data_files", []):
        file_path = key_dir / file_name

        if file_path.suffix.lower() != ".csv":
            non_csv_paths.append(file_path)
            print(f"Non-CSV file in data_files: {file_path}")
            continue

        if not file_path.exists():
            missing_paths.append(file_path)
            print(f"Missing file: {file_path}")
            continue

        try:
            pd.read_csv(file_path, nrows=3)
        except Exception as exc:
            broken_csv_paths.append(file_path)
            print(f"Broken CSV file: {file_path}")
            print(f"  Reason: {exc}")

print(f"\nChecked keys: {len(all_keys)}")
print(f"Missing files: {len(missing_paths)}")
print(f"Non-CSV files: {len(non_csv_paths)}")
print(f"Broken CSV files: {len(broken_csv_paths)}")


Checked keys: 760
Missing files: 0
Non-CSV files: 0
Broken CSV files: 0


In [30]:
# Process one folder (one key): detect SMILES columns in CSV files and store `ess_columns`
import pandas as pd

KEY_ONE = "ASPECT"  # change key here
key_dir = Path(WORKSPACE) / KEY_ONE

detect_files_state, _ = VersionedYaml(key_dir).read("detect_files")
csv_files = [f for f in detect_files_state.get("data_files", []) if str(f).lower().endswith(".csv")]

smiles_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a chemistry data assistant. Find CSV columns that contain SMILES strings.",
    ),
    (
        "user",
        """
Return JSON matching this schema only:
- SMILES_columns: list[str]

Rules:
- Pick only columns present in the preview.
- If none look like SMILES, return an empty list.
- Do not invent column names.

CSV file name:
{input_csv}

CSV preview (header + up to 2 rows):
{csv_preview}

Validation error from previous attempt (empty for first try):
{validation_error}
""".strip(),
    ),
])

structured_llm = LLM_BASIC.with_structured_output(SMILES_Data)
file_and_columns: list[SMILESandCSV_Data] = []

for file_name in csv_files:
    file_path = key_dir / file_name

    if not file_path.exists():
        print(f"Missing file: {file_path}")
        continue

    try:
        preview_df = pd.read_csv(file_path, nrows=2)
    except Exception as exc:
        print(f"Broken CSV file: {file_path}")
        print(f"  Reason: {exc}")
        continue

    csv_preview = preview_df.to_csv(index=False)
    valid_columns = list(preview_df.columns)

    validation_error = ""
    llm_result = SMILES_Data(SMILES_columns=[])

    for attempt in range(1, RETRY + 1):
        messages = smiles_prompt.format_messages(
            input_csv=file_name,
            csv_preview=csv_preview,
            validation_error=validation_error,
        )
        llm_result = structured_llm.invoke(messages, config=LF_CONFIG)

        invalid_columns = [c for c in llm_result.SMILES_columns if c not in valid_columns]
        if not invalid_columns:
            break

        validation_error = (
            f"Invalid columns returned: {invalid_columns}. "
            f"Allowed columns are: {valid_columns}."
        )
        if attempt == RETRY:
            print(f"LLM validation failed for {file_path}")
            print(f"  {validation_error}")
            llm_result = SMILES_Data(SMILES_columns=[])

    # Manual object construction from validated LLM output
    unique_columns = list(dict.fromkeys(llm_result.SMILES_columns))
    file_and_columns.append(
        SMILESandCSV_Data(
            CSV_file_name=file_name,
            SMILES_columns=unique_columns,
        )
    )

ess_columns = EssentialColumnAnalysis(file_and_columns=file_and_columns)
VersionedYaml(key_dir).create("ess_columns", ess_columns.model_dump())
ess_columns

Failed to detach context
Traceback (most recent call last):
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/langfuse/_client/propagation.py", line 272, in _propagate_attributes
    yield
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/opentelemetry/context/contextvars_context.py", line 53, in detach
    self._current_context.reset(token)
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x72a347f7ba60> at 0x72a30216a940> was created in a different Context


EssentialColumnAnalysis(file_and_columns=[SMILESandCSV_Data(SMILES_columns=['smiles'], CSV_file_name='chemicals.csv')])

In [31]:
KEY = "ASPECT" #"ALGALTOX"
d,_ = VersionedYaml(Path(WORKSPACE) / KEY).read("ess_columns")
d

{'file_and_columns': [{'SMILES_columns': ['smiles'],
   'CSV_file_name': 'chemicals.csv'}]}

In [32]:
# # Run SMILES detection for all workspace keys (fault-tolerant like in step02)
# import traceback
# import pandas as pd

# smiles_prompt = ChatPromptTemplate.from_messages([
#     (
#         "system",
#         "You are a chemistry data assistant. Find CSV columns that contain SMILES strings.",
#     ),
#     (
#         "user",
#         """
# Return JSON matching this schema only:
# - SMILES_columns: list[str]

# Rules:
# - Pick only columns present in the preview.
# - If none look like SMILES, return an empty list.
# - Do not invent column names.

# CSV file name:
# {input_csv}

# CSV preview (header + up to 2 rows):
# {csv_preview}

# Validation error from previous attempt (empty for first try):
# {validation_error}
# """.strip(),
#     ),
# ])
# structured_llm = LLM_BASIC.with_structured_output(SMILES_Data)

# all_keys = sorted(
#     p.name
#     for p in Path(WORKSPACE).iterdir()
#     if p.is_dir() and not p.name.startswith(".")
# )

# batch_ess_results: list[dict] = []

# for key in all_keys:
#     try:
#         key_dir = Path(WORKSPACE) / key
#         detect_files_state, _ = VersionedYaml(key_dir).read("detect_files")
#         csv_files = [f for f in detect_files_state.get("data_files", []) if str(f).lower().endswith(".csv")]

#         file_and_columns: list[SMILESandCSV_Data] = []

#         for file_name in csv_files:
#             file_path = key_dir / file_name
#             if not file_path.exists():
#                 continue

#             try:
#                 preview_df = pd.read_csv(file_path, nrows=2)
#             except Exception:
#                 continue

#             valid_columns = list(preview_df.columns)
#             csv_preview = preview_df.to_csv(index=False)
#             validation_error = ""
#             llm_result = SMILES_Data(SMILES_columns=[])

#             for attempt in range(1, RETRY + 1):
#                 messages = smiles_prompt.format_messages(
#                     input_csv=file_name,
#                     csv_preview=csv_preview,
#                     validation_error=validation_error,
#                 )
#                 llm_result = structured_llm.invoke(messages, config=LF_CONFIG)

#                 invalid_columns = [c for c in llm_result.SMILES_columns if c not in valid_columns]
#                 if not invalid_columns:
#                     break

#                 validation_error = (
#                     f"Invalid columns returned: {invalid_columns}. "
#                     f"Allowed columns are: {valid_columns}."
#                 )
#                 if attempt == RETRY:
#                     llm_result = SMILES_Data(SMILES_columns=[])

#             unique_columns = list(dict.fromkeys(llm_result.SMILES_columns))
#             file_and_columns.append(
#                 SMILESandCSV_Data(
#                     CSV_file_name=file_name,
#                     SMILES_columns=unique_columns,
#                 )
#             )

#         ess_columns = EssentialColumnAnalysis(file_and_columns=file_and_columns)
#         VersionedYaml(key_dir).create("ess_columns", ess_columns.model_dump())

#         batch_ess_results.append({"key": key, "ok": True, "state": ess_columns})
#         print(f"OK  {key}")

#     except Exception as e:
#         batch_ess_results.append(
#             {
#                 "key": key,
#                 "ok": False,
#                 "error": repr(e),
#                 "traceback": traceback.format_exc(),
#             }
#         )
#         print(f"FAIL {key}: {e!r}")

# failed_keys = [r["key"] for r in batch_ess_results if not r["ok"]]
# print(f"\nDone: {len(batch_ess_results)} keys, {len(failed_keys)} failed")
# failed_keys

In [33]:
# LangGraph subgraph: process one key (async, with logging enabled)
from typing import Annotated
import operator
import pandas as pd

from langgraph.types import Send
from langchain_core.runnables import RunnableConfig


def _cfg(base: dict, **updates) -> RunnableConfig:
    cfg = dict(base or {})
    if isinstance(cfg.get("callbacks"), list):
        cfg["callbacks"] = list(cfg["callbacks"])
    cfg.update(updates)
    return RunnableConfig(**cfg)


smiles_columns_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a chemistry data assistant. Find CSV columns that contain SMILES strings.",
    ),
    (
        "user",
        """
Return JSON matching this schema only:
- SMILES_columns: list[str]

Rules:
- Pick only columns present in the preview.
- If none look like SMILES, return an empty list.
- Do not invent column names.

CSV file name:
{input_csv}

CSV preview (header + up to 2 rows):
{csv_preview}

Validation error from previous attempt (empty for first try):
{validation_error}
""".strip(),
    ),
])


class OneKeySMILESState(BaseModel):
    workspace_path: str
    key: str
    csv_files: List[str] = Field(default_factory=list)
    file_and_columns: List[SMILESandCSV_Data] = Field(default_factory=list)
    ess_columns: Optional[EssentialColumnAnalysis] = None


async def _detect_smiles_columns_for_csv(file_name: str, preview_df: pd.DataFrame) -> list[str]:
    structured_llm = LLM_BASIC.with_structured_output(SMILES_Data)
    valid_columns = list(preview_df.columns)
    csv_preview = preview_df.to_csv(index=False)

    validation_error = ""
    llm_result = SMILES_Data(SMILES_columns=[])

    for _ in range(RETRY):
        messages = smiles_columns_prompt.format_messages(
            input_csv=file_name,
            csv_preview=csv_preview,
            validation_error=validation_error,
        )
        llm_result = await structured_llm.ainvoke(messages, config=_cfg(LF_CONFIG))

        invalid_columns = [c for c in llm_result.SMILES_columns if c not in valid_columns]
        if not invalid_columns:
            return list(dict.fromkeys(llm_result.SMILES_columns))

        validation_error = (
            f"Invalid columns returned: {invalid_columns}. "
            f"Allowed columns are: {valid_columns}."
        )

    return []


def load_key_csv_files(state: OneKeySMILESState) -> dict:
    key_dir = Path(state.workspace_path) / state.key
    detect_files_state, _ = VersionedYaml(key_dir).read("detect_files")
    csv_files = [f for f in detect_files_state.get("data_files", []) if str(f).lower().endswith(".csv")]
    return {"csv_files": csv_files}


async def detect_key_smiles_columns(state: OneKeySMILESState) -> dict:
    key_dir = Path(state.workspace_path) / state.key
    file_and_columns: list[SMILESandCSV_Data] = []

    for file_name in state.csv_files:
        file_path = key_dir / file_name
        if not file_path.exists():
            continue

        try:
            preview_df = pd.read_csv(file_path, nrows=2)
        except Exception:
            continue

        smiles_columns = await _detect_smiles_columns_for_csv(file_name, preview_df)
        file_and_columns.append(
            SMILESandCSV_Data(
                CSV_file_name=file_name,
                SMILES_columns=smiles_columns,
            )
        )

    return {"file_and_columns": file_and_columns}


def store_key_ess_columns(state: OneKeySMILESState) -> dict:
    key_dir = Path(state.workspace_path) / state.key
    ess_columns = EssentialColumnAnalysis(file_and_columns=state.file_and_columns)
    VersionedYaml(key_dir).create("ess_columns", ess_columns.model_dump())
    return {"ess_columns": ess_columns}


one_key_smiles_graph = StateGraph(OneKeySMILESState)
one_key_smiles_graph.add_node("load_key_csv_files", load_key_csv_files)
one_key_smiles_graph.add_node("detect_key_smiles_columns", detect_key_smiles_columns)
one_key_smiles_graph.add_node("store_key_ess_columns", store_key_ess_columns)

one_key_smiles_graph.add_edge(START, "load_key_csv_files")
one_key_smiles_graph.add_edge("load_key_csv_files", "detect_key_smiles_columns")
one_key_smiles_graph.add_edge("detect_key_smiles_columns", "store_key_ess_columns")
one_key_smiles_graph.add_edge("store_key_ess_columns", END)

one_key_smiles_app = one_key_smiles_graph.compile()

In [34]:
# LangGraph parent graph: process all keys (uses one-key subgraph)
class BatchSMILESState(BaseModel):
    workspace_path: str
    keys: List[str]
    results: Annotated[List[dict], operator.add] = Field(default_factory=list)


def dispatch_smiles_keys(state: BatchSMILESState):
    return [
        Send(
            "process_one_smiles_key",
            {
                "workspace_path": state.workspace_path,
                "key": key,
            },
        )
        for key in state.keys
    ]


async def process_one_smiles_key(state) -> dict:
    key = state["key"]
    workspace_path = state["workspace_path"]

    try:
        raw = await one_key_smiles_app.ainvoke(
            OneKeySMILESState(workspace_path=workspace_path, key=key),
            config=_cfg(LF_CONFIG),
        )
        final = raw if isinstance(raw, OneKeySMILESState) else OneKeySMILESState.model_validate(raw)
        return {
            "results": [
                {
                    "key": key,
                    "ok": True,
                    "csv_files": len(final.csv_files),
                    "records": len(final.file_and_columns),
                }
            ]
        }
    except Exception as e:
        return {
            "results": [
                {
                    "key": key,
                    "ok": False,
                    "error": repr(e),
                }
            ]
        }


batch_smiles_graph = StateGraph(BatchSMILESState)
batch_smiles_graph.add_node("process_one_smiles_key", process_one_smiles_key)

batch_smiles_graph.add_conditional_edges(START, dispatch_smiles_keys)
batch_smiles_graph.add_edge("process_one_smiles_key", END)

batch_smiles_app = batch_smiles_graph.compile()

In [35]:
# Run full SMILES processing for all keys
all_keys = sorted(
    p.name
    for p in Path(WORKSPACE).iterdir()
    if p.is_dir() and not p.name.startswith(".")
)

initial_state = BatchSMILESState(
    workspace_path=WORKSPACE,
    keys=all_keys,
)

run_config = _cfg(LF_CONFIG, max_concurrency=2)
raw = await batch_smiles_app.ainvoke(initial_state.model_dump(), config=run_config)
final_batch = raw if isinstance(raw, BatchSMILESState) else BatchSMILESState.model_validate(raw)

failed_keys = [r["key"] for r in final_batch.results if not r.get("ok", False)]
print(f"Done: {len(final_batch.results)} keys, {len(failed_keys)} failed")
failed_keys

Failed to detach context
Traceback (most recent call last):
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/langfuse/_client/propagation.py", line 272, in _propagate_attributes
    yield
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/opentelemetry/context/contextvars_context.py", line 53, in detach
    self._current_context.reset(token)
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x72a347f7ba60> at 0x72a303749880> was created in a different Context
Propagated attribute 'metadata.langgraph_step' value is not a string. Dropping value.
'_AgnosticContextManager' object has no attribute 'args'
Traceback (most recent call last):
  File 

Done: 760 keys, 44 failed


['Melting_point',
 'MetalCytoToxDB',
 'Micronucleus_DB',
 'MixtureSolDB',
 'MolecilarML2034_DB',
 'MolecilarML262_DB',
 'MolecularML2047_DB',
 'MolecularML231_DB',
 'MolecularML238_DB',
 'MolecularML264_DB',
 'MolecularML2835_DB',
 'MolecularML2971_DB',
 'MolecularML4203_DB',
 'MolecularML4616_DB',
 'NANJINGOPDE',
 'NAPHTHA',
 'NATADB',
 'NATOXAQ',
 'NDATAPEST',
 'NDSEXTRAS',
 'NERVEAGENTS',
 'NEUROTOXINS',
 'NHANES_DKAPRAUN',
 'NIOSHICSC',
 'NIOSHNPG',
 'NIOSHSKIN',
 'NISTSRM1950',
 'NONYLPHENOLS',
 'PFASFCCMIGEX',
 'PFASGLUEGE',
 'PFASINVITRO',
 'PFASKEMI',
 'PFASMARKUSH',
 'PFASMASSBANK',
 'PFASMASTER',
 'PFASMONA',
 'PFASNTREV19',
 'PFASOECD',
 'PFASOECDNA',
 'PFASPACKAGING',
 'PFASSTRUCT',
 'PFASSTRUCTV1',
 'PFASSTRUCTV2',
 'PFASSTRUCTV3']

In [36]:
failed_keys

['Melting_point',
 'MetalCytoToxDB',
 'Micronucleus_DB',
 'MixtureSolDB',
 'MolecilarML2034_DB',
 'MolecilarML262_DB',
 'MolecularML2047_DB',
 'MolecularML231_DB',
 'MolecularML238_DB',
 'MolecularML264_DB',
 'MolecularML2835_DB',
 'MolecularML2971_DB',
 'MolecularML4203_DB',
 'MolecularML4616_DB',
 'NANJINGOPDE',
 'NAPHTHA',
 'NATADB',
 'NATOXAQ',
 'NDATAPEST',
 'NDSEXTRAS',
 'NERVEAGENTS',
 'NEUROTOXINS',
 'NHANES_DKAPRAUN',
 'NIOSHICSC',
 'NIOSHNPG',
 'NIOSHSKIN',
 'NISTSRM1950',
 'NONYLPHENOLS',
 'PFASFCCMIGEX',
 'PFASGLUEGE',
 'PFASINVITRO',
 'PFASKEMI',
 'PFASMARKUSH',
 'PFASMASSBANK',
 'PFASMASTER',
 'PFASMONA',
 'PFASNTREV19',
 'PFASOECD',
 'PFASOECDNA',
 'PFASPACKAGING',
 'PFASSTRUCT',
 'PFASSTRUCTV1',
 'PFASSTRUCTV2',
 'PFASSTRUCTV3']

In [37]:
# Run full SMILES processing for all keys
all_keys = failed_keys

initial_state = BatchSMILESState(
    workspace_path=WORKSPACE,
    keys=all_keys,
)

run_config = _cfg(LF_CONFIG, max_concurrency=2)
raw = await batch_smiles_app.ainvoke(initial_state.model_dump(), config=run_config)
final_batch = raw if isinstance(raw, BatchSMILESState) else BatchSMILESState.model_validate(raw)

failed_keys = [r["key"] for r in final_batch.results if not r.get("ok", False)]
print(f"Done: {len(final_batch.results)} keys, {len(failed_keys)} failed")
failed_keys

Failed to detach context
Traceback (most recent call last):
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/langfuse/_client/propagation.py", line 272, in _propagate_attributes
    yield
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/home/kostanew/2026/CDGReview/.venv/lib/python3.11/site-packages/opentelemetry/context/contextvars_context.py", line 53, in detach
    self._current_context.reset(token)
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x72a347f7ba60> at 0x72a30213e1c0> was created in a different Context
Propagated attribute 'metadata.langgraph_step' value is not a string. Dropping value.
'_AgnosticContextManager' object has no attribute 'args'
Traceback (most recent call last):
  File 

Done: 44 keys, 0 failed


[]